# Task 2: SQL for Data Extraction

**ApexPlanet Software Pvt. Ltd. — Data Analytics Internship**

**Objective:** Master SQL queries for data extraction and business analysis.

In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/ecommerce_sales_cleaned.csv")

# Create SQLite database
conn = sqlite3.connect("../data/ecommerce.db")
df.to_sql("sales", conn, if_exists="replace", index=False)

print("✅ Loaded", len(df), "rows into SQLite database!")
print("Table: sales")

## Day 7-8: SQL Fundamentals

We'll use a helper function to run SQL queries easily:

In [1]:
def run_sql(query, label=""):
    """Run a SQL query and return results as a DataFrame"""
    result = pd.read_sql(query, conn)
    if label:
        print(f"--- {label} ---")
    return result

In [1]:
# SELECT, WHERE, ORDER BY
run_sql("""
SELECT Order_ID, Category, Region, Net_Sales
FROM sales
WHERE Net_Sales > 5000
ORDER BY Net_Sales DESC
LIMIT 10
""", "Top 10 orders above ₹5000")

In [1]:
# GROUP BY + ORDER BY - Sales by Category
run_sql("""
SELECT Category,
       COUNT(*) AS total_orders,
       ROUND(SUM(Net_Sales), 2) AS total_sales,
       ROUND(AVG(Net_Sales), 2) AS avg_order_value
FROM sales
GROUP BY Category
ORDER BY total_sales DESC
""", "Sales by Category")

In [1]:
# HAVING - Customers with more than 8 orders
run_sql("""
SELECT Customer_ID, COUNT(*) AS order_count, ROUND(SUM(Net_Sales),2) AS total_spend
FROM sales
GROUP BY Customer_ID
HAVING order_count > 8
ORDER BY total_spend DESC
LIMIT 10
""", "High-frequency customers (>8 orders)")

## Day 9-10: Advanced SQL — CTEs and Window Functions

In [1]:
# CTE - Top customers
run_sql("""
WITH customer_totals AS (
    SELECT Customer_ID,
           SUM(Net_Sales) AS total_spend,
           COUNT(*) AS orders
    FROM sales
    GROUP BY Customer_ID
)
SELECT * FROM customer_totals
ORDER BY total_spend DESC
LIMIT 10
""", "Top 10 Customers by Spend (CTE)")

In [1]:
# Window Function - RANK top products per category
run_sql("""
SELECT Category, Product_Name, total_sales, rank_in_category
FROM (
    SELECT Category, Product_Name, total_sales,
           RANK() OVER (PARTITION BY Category ORDER BY total_sales DESC) AS rank_in_category
    FROM (
        SELECT Category, Product_Name, SUM(Net_Sales) AS total_sales
        FROM sales
        GROUP BY Category, Product_Name
    )
)
WHERE rank_in_category <= 2
ORDER BY Category, rank_in_category
""", "Top 2 Products per Category (RANK window function)")

In [1]:
# Window Function - LAG for month-over-month change
run_sql("""
SELECT Order_Month, monthly_sales,
       LAG(monthly_sales) OVER (ORDER BY Order_Month) AS prev_month,
       ROUND(monthly_sales - LAG(monthly_sales) OVER (ORDER BY Order_Month), 2) AS mom_change
FROM (
    SELECT Order_Month, ROUND(SUM(Net_Sales),2) AS monthly_sales
    FROM sales
    GROUP BY Order_Month
)
ORDER BY Order_Month
""", "Month-over-Month Sales Change (LAG)")

In [1]:
# Create a reusable VIEW
conn.execute("""
    CREATE VIEW IF NOT EXISTS v_category_performance AS
    SELECT Category,
           COUNT(*) AS total_orders,
           ROUND(SUM(Net_Sales), 2) AS total_sales,
           ROUND(AVG(Net_Sales), 2) AS avg_order_value,
           ROUND(SUM(Profit), 2) AS total_profit
    FROM sales GROUP BY Category
""")
conn.commit()

run_sql("SELECT * FROM v_category_performance ORDER BY total_sales DESC", "Category Performance View")

## Day 11-13: Python + SQL — 10 Business Questions

In [1]:
run_sql("""
SELECT Product_Name, ROUND(SUM(Net_Sales),2) AS total_sales FROM sales GROUP BY Product_Name ORDER BY total_sales DESC LIMIT 5
""", "Q1: Top 5 Products by Sales")

In [1]:
run_sql("""
SELECT Order_Month, ROUND(SUM(Net_Sales),2) AS total_sales FROM sales GROUP BY Order_Month ORDER BY Order_Month
""", "Q2: Monthly Sales Trend")

In [1]:
run_sql("""
SELECT Customer_Segment, COUNT(DISTINCT Customer_ID) AS customers, ROUND(SUM(Net_Sales),2) AS total_sales, ROUND(AVG(Net_Sales),2) AS avg_order_value FROM sales GROUP BY Customer_Segment ORDER BY total_sales DESC
""", "Q3: Customer Segmentation by Spend")

In [1]:
run_sql("""
SELECT Region, ROUND(SUM(Profit),2) AS total_profit FROM sales GROUP BY Region ORDER BY total_profit DESC
""", "Q4: Top Region by Profit")

In [1]:
run_sql("""
SELECT Payment_Method, COUNT(*) AS orders, ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM sales),1) AS pct FROM sales GROUP BY Payment_Method ORDER BY orders DESC
""", "Q5: Payment Method Popularity")

In [1]:
run_sql("""
SELECT Category, ROUND(AVG(Discount_Percent),2) AS avg_discount FROM sales GROUP BY Category ORDER BY avg_discount DESC
""", "Q6: Avg Discount by Category")

In [1]:
run_sql("""
SELECT CASE WHEN order_count=1 THEN 'One-time' ELSE 'Repeat' END AS type, COUNT(*) AS customers, ROUND(SUM(total_spend),2) AS total_spend FROM (SELECT Customer_ID, COUNT(*) AS order_count, SUM(Net_Sales) AS total_spend FROM sales GROUP BY Customer_ID) GROUP BY type
""", "Q7: Repeat vs One-time Customers")

In [1]:
run_sql("""
SELECT Order_Weekday, ROUND(SUM(Net_Sales),2) AS total_sales, COUNT(*) AS orders FROM sales GROUP BY Order_Weekday ORDER BY total_sales DESC
""", "Q8: Best Weekday for Sales")

In [1]:
run_sql("""
SELECT Region, Ship_Mode, COUNT(*) AS orders FROM sales GROUP BY Region, Ship_Mode ORDER BY Region, orders DESC
""", "Q9: Shipping Mode by Region")

In [1]:
run_sql("""
SELECT Order_ID, Customer_ID, Category, Net_Sales FROM sales WHERE Net_Sales > (SELECT AVG(Net_Sales) FROM sales) ORDER BY Net_Sales DESC LIMIT 10
""", "Q10: High-Value Orders Above Average")

In [1]:
# Reusable utility function
def query_db(sql):
    """Reusable helper to query the sales database"""
    return pd.read_sql(sql, conn)

print("✅ Task 2 Complete! All 10 business questions answered with SQL.")
conn.close()